# Titanic Survivors — Consolidated Notebook

This notebook consolidates the strongest parts of `titanic_eda` and `titanic_improved`:
1. Robust, train-fitted preprocessing to reduce leakage
2. Expanded feature set (bins, deck, ticket frequency, interactions)
3. Accuracy-oriented model selection for Kaggle Titanic
4. Final refit on all labeled data + strict submission format


## 1  Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score
import random
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.model_selection import GridSearchCV

%matplotlib inline
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


## 2  Load Data

> Place `train.csv`, `test.csv`, and `gender_submission.csv` inside the `data/` folder.  
> See `data/README.md` for download instructions.

In [ ]:
train_raw = pd.read_csv('data/train.csv')
test_raw  = pd.read_csv('data/test.csv')

print(f'Training set : {train_raw.shape[0]} rows × {train_raw.shape[1]} columns')
print(f'Test set     : {test_raw.shape[0]} rows × {test_raw.shape[1]} columns')
train_raw.head()

## 3  Exploratory Data Analysis

In [ ]:
train_raw.info()

In [ ]:
train_raw.describe(include='all')

In [ ]:
# Missing-value summary
missing = train_raw.isnull().sum().rename('missing').to_frame()
missing['pct'] = (missing['missing'] / len(train_raw) * 100).round(2)
missing[missing['missing'] > 0].sort_values('pct', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Survival rate
train_raw['Survived'].value_counts().plot(kind='bar', ax=axes[0], color=['#d9534f', '#5cb85c'])
axes[0].set_title('Survival Counts')
axes[0].set_xticklabels(['Died (0)', 'Survived (1)'], rotation=0)

# Survival by Sex
train_raw.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[1], color=['#5bc0de', '#f0ad4e'])
axes[1].set_title('Survival Rate by Sex')
axes[1].set_ylabel('Survival Rate')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

# Survival by Pclass
train_raw.groupby('Pclass')['Survived'].mean().plot(kind='bar', ax=axes[2], color='steelblue')
axes[2].set_title('Survival Rate by Pclass')
axes[2].set_ylabel('Survival Rate')
axes[2].set_xticklabels(['1st', '2nd', '3rd'], rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Age distribution
train_raw['Age'].plot(kind='hist', bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')

# Fare distribution
train_raw['Fare'].plot(kind='hist', bins=40, ax=axes[1], color='teal', edgecolor='white')
axes[1].set_title('Fare Distribution')
axes[1].set_xlabel('Fare')

plt.tight_layout()
plt.show()

## 4  Feature Engineering

We apply the same transformations to both the training and test sets to avoid data leakage and ensure consistency.

| Step | Action |
|------|--------|
| `Title` | Extract title from passenger name as a social-status proxy |
| `FamilySize` | `SibSp + Parch + 1` — total people in family unit |
| `IsAlone` | Binary flag: 1 if travelling alone, else 0 |
| `Age` | Impute missing values with the median age per title group |
| `Fare` | Impute one missing test value with the overall median |
| `Embarked` | Impute two missing values with the mode (`S`) |
| Encoding | Label-encode `Sex`; one-hot-encode `Embarked` and `Title` |
| Drop | Remove `PassengerId`, `Name`, `Ticket`, `Cabin` (high cardinality / mostly missing) |

In [ ]:
def add_base_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create raw engineered columns before imputation/encoding."""
    out = df.copy()

    # --- Title -----------------------------------------------------------
    out['Title'] = out['Name'].str.extract(r',\s*([^.]+)\.', expand=False).str.strip()
    rare_titles = {'Dona', 'Lady', 'Countess', 'the Countess', 'Capt', 'Col', 'Don',
                   'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer'}
    out['Title'] = out['Title'].replace(list(rare_titles), 'Rare')
    out['Title'] = out['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

    # --- Family size + interaction features -----------------------------
    out['FamilySize'] = out['SibSp'] + out['Parch'] + 1
    out['IsAlone'] = (out['FamilySize'] == 1).astype(int)
    out['Sex'] = out['Sex'].map({'male': 0, 'female': 1}).fillna(0).astype(int)
    out['Sex_Pclass'] = out['Sex'] * out['Pclass']

    # --- Deck and ticket group size proxy -------------------------------
    out['Deck'] = out['Cabin'].str[0].fillna('U')
    return out


def fit_preprocessor(train_df: pd.DataFrame) -> dict:
    """Fit preprocessing statistics on train data only."""
    t = add_base_features(train_df)

    # Ticket frequency fitted on train only.
    ticket_freq_map = t['Ticket'].value_counts().to_dict()

    # Fare bins from train quantiles, stabilized for duplicates.
    fare_values = t['Fare'].fillna(t['Fare'].median())
    fare_edges = np.unique(fare_values.quantile([0.0, 0.25, 0.5, 0.75, 1.0]).to_numpy())
    if len(fare_edges) < 2:
        fare_edges = np.array([fare_values.min() - 1.0, fare_values.max() + 1.0])
    fare_edges = fare_edges.astype(float)
    fare_edges[0] -= 1e-9
    fare_edges[-1] += 1e-9
    fare_labels = [f'FareBin_{i+1}' for i in range(len(fare_edges) - 1)]

    prep = {
        'title_age_medians': t.groupby('Title')['Age'].median(),
        'age_median': t['Age'].median(),
        'fare_median': t['Fare'].median(),
        'embarked_mode': t['Embarked'].mode()[0],
        'ticket_freq_map': ticket_freq_map,
        'fare_edges': fare_edges,
        'fare_labels': fare_labels,
        'title_levels': sorted(t['Title'].dropna().unique().tolist()),
        'embarked_levels': sorted(t['Embarked'].dropna().unique().tolist()),
        'deck_levels': sorted(t['Deck'].dropna().unique().tolist()),
        'agebin_levels': ['Child', 'Teen', 'Adult', 'Senior'],
        'familysizebin_levels': ['Solo', 'Small', 'Large'],
    }
    return prep


def transform_features(df: pd.DataFrame, prep: dict) -> pd.DataFrame:
    """Apply train-fitted preprocessing stats to any dataframe."""
    out = add_base_features(df)

    # Train-fitted imputations
    out['Age'] = out['Age'].fillna(out['Title'].map(prep['title_age_medians']))
    out['Age'] = out['Age'].fillna(prep['age_median'])
    out['Fare'] = out['Fare'].fillna(prep['fare_median'])
    out['Embarked'] = out['Embarked'].fillna(prep['embarked_mode'])

    # Additional engineered features
    out['TicketFreq'] = out['Ticket'].map(prep['ticket_freq_map']).fillna(1).astype(int)

    out['AgeBin'] = pd.cut(
        out['Age'], bins=[0, 12, 18, 60, 120], labels=prep['agebin_levels'], include_lowest=True
    )

    out['FamilySizeBin'] = pd.cut(
        out['FamilySize'], bins=[0, 1, 4, 100], labels=prep['familysizebin_levels'], include_lowest=True
    )

    out['FareBin'] = pd.cut(
        out['Fare'], bins=prep['fare_edges'], labels=prep['fare_labels'], include_lowest=True
    )

    # Freeze category levels to train levels
    out['Embarked'] = pd.Categorical(out['Embarked'], categories=prep['embarked_levels'])
    out['Title'] = pd.Categorical(out['Title'], categories=prep['title_levels'])
    out['Deck'] = pd.Categorical(out['Deck'], categories=prep['deck_levels'])
    out['AgeBin'] = pd.Categorical(out['AgeBin'], categories=prep['agebin_levels'])
    out['FamilySizeBin'] = pd.Categorical(out['FamilySizeBin'], categories=prep['familysizebin_levels'])
    out['FareBin'] = pd.Categorical(out['FareBin'], categories=prep['fare_labels'])

    out = pd.get_dummies(
        out,
        columns=['Embarked', 'Title', 'Deck', 'AgeBin', 'FareBin', 'FamilySizeBin'],
        drop_first=False,
    )

    drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
    out = out.drop(columns=[c for c in drop_cols if c in out.columns])
    return out


# Full-data preprocessor for final model fit/submission generation
preprocessor = fit_preprocessor(train_raw)
train_eng = transform_features(train_raw, preprocessor)
test_eng = transform_features(test_raw, preprocessor)

# Align test columns to training columns (excluding target)
feature_cols = [c for c in train_eng.columns if c != 'Survived']
test_eng = test_eng.reindex(columns=feature_cols, fill_value=0)

print('Engineered training set shape :', train_eng.shape)
print('Engineered test set shape     :', test_eng.shape)
train_eng.head()


In [ ]:
# Confirm no missing values remain in training set
missing_after = train_eng.isnull().sum()
print('Columns still containing NaN:', missing_after[missing_after > 0].to_dict() or 'None ✓')

## 5  Train / Validation Split (80 / 20)

We use stratified splits to preserve class distribution across train/validation/tuning/holdout.


In [ ]:
X = train_raw.drop(columns=['Survived'])
y = train_raw['Survived']

raw_train_df, raw_val_df = train_test_split(
    train_raw,
    test_size=0.20,
    stratify=train_raw['Survived'],
    random_state=42
)

split_preprocessor = fit_preprocessor(raw_train_df)
train_df = transform_features(raw_train_df, split_preprocessor)
val_df = transform_features(raw_val_df, split_preprocessor)

model_feature_cols = [c for c in train_df.columns if c != 'Survived']
X_train, y_train = train_df[model_feature_cols].values, train_df['Survived'].values
X_val, y_val = val_df.reindex(columns=model_feature_cols, fill_value=0).values, val_df['Survived'].values

# Split validation into threshold-tuning and final holdout subsets (stratified)
X_thr, X_holdout, y_thr, y_holdout = train_test_split(
    X_val,
    y_val,
    test_size=0.50,
    stratify=y_val,
    random_state=42
)

print(f"Train {len(train_df):,} | Val {len(val_df):,}")
print(f'X_train : {X_train.shape}  |  y_train : {y_train.shape}')
print(f'X_val   : {X_val.shape}   |  y_val   : {y_val.shape}')
print(f'X_thr   : {X_thr.shape}   |  y_thr   : {y_thr.shape}')
print(f'X_holdout: {X_holdout.shape} | y_holdout: {y_holdout.shape}')
print()
print('Survival rate — train :', round(y_train.mean(), 4))
print('Survival rate — val   :', round(y_val.mean(), 4))
print('Survival rate — thr   :', round(y_thr.mean(), 4))
print('Survival rate — holdout:', round(y_holdout.mean(), 4))
print()
print('Features used:')
print(model_feature_cols)


## Summary

| Dataset | Rows | Features |
|---------|------|----------|
| Full training (raw) | 891 | 12 |
| Engineered training | 891 | varies |
| **X_train** | ~713 | — |
| **X_val** | ~178 | — |
| Test (Kaggle) | 418 | — |

The splits `X_train`, `X_val`, `y_train`, `y_val` and the Kaggle test features `test_eng` are ready for model training in a follow-up notebook.

## 7 Setup XGBOOST

In [ ]:
np.random.seed(42)
random.seed(42)

In [ ]:
param_grid = {
    'n_estimators': [300, 600],
    'max_depth': [3, 4],
    'learning_rate': [0.01, 0.03, 0.05],
    'subsample': [0.7, 0.8],
    'colsample_bytree': [0.7, 0.8],
    'scale_pos_weight': [1.0],
    'gamma': [0.0, 0.1],
    'min_child_weight': [1, 3],
    'reg_alpha': [0.0, 0.1],
    'reg_lambda': [1.0, 2.0],
}

search_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    random_state=42
)

# Kaggle Titanic leaderboard metric is accuracy
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    estimator=search_model,
    param_grid=param_grid,
    scoring='accuracy',
    cv=skf,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

# Threshold tuning on dedicated tuning split (optimize accuracy)
thr_proba = best_model.predict_proba(X_thr)[:, 1]
thresholds = np.arange(0.25, 0.76, 0.01)
acc_scores = [accuracy_score(y_thr, (thr_proba >= t).astype(int)) for t in thresholds]
best_thr = float(thresholds[int(np.argmax(acc_scores))])

# Final metric on untouched holdout split
holdout_proba = best_model.predict_proba(X_holdout)[:, 1]
y_pred = (holdout_proba >= best_thr).astype(int)

acc = accuracy_score(y_holdout, y_pred)
f1 = f1_score(y_holdout, y_pred)
cm = confusion_matrix(y_holdout, y_pred)

print(f'Best CV accuracy: {grid_search.best_score_:.4f}')
print(f'Best CV params: {grid_search.best_params_}')
print(f'Best threshold on tuning split: {best_thr:.3f}')
print(f'Accuracy on Holdout Set: {acc:.4f}')
print(f'F1 Score on Holdout Set: {f1:.4f}')
print(f'Confusion Matrix on Holdout Set:\n{cm}')


## 8  Create Kaggle Submission

In [ ]:
# Refit best hyperparameters on all labeled data before final submission
X_full = train_eng[feature_cols].values
y_full = train_eng['Survived'].values
final_model = XGBClassifier(**best_model.get_params())
final_model.fit(X_full, y_full)

# Predict on Kaggle test features using tuned threshold
test_proba = final_model.predict_proba(test_eng.values)[:, 1]
test_pred = (test_proba >= best_thr).astype(int)

submission = pd.DataFrame({
    'PassengerId': test_raw['PassengerId'].values,
    'Survived': test_pred
})

# Enforce required submission schema
submission = submission[['PassengerId', 'Survived']]
assert list(submission.columns) == ['PassengerId', 'Survived']
assert len(submission) == 418
assert submission['Survived'].isin([0, 1]).all()

submission.to_csv('submission_consolidated.csv', index=False)
print('Saved submission_consolidated.csv')
print('Shape:', submission.shape)
print(f'Survival rate in submission: {submission["Survived"].mean():.3f}')
submission.head()
